# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # keep as an object, not subscripting

# Print overview
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata.id}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Personal Sensitive Information: {getattr(metadata, 'personalSensitiveInformation', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

First, list all record sets, fields, and columns by their `@id`. Then examine their structure for subsequent extraction.

In [ ]:
# Discover available record sets and their @id
record_sets_info = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets_info:
    print("No record sets found in metadata. Attempting to load records...")
    # mlcroissant's API: dataset.records() lists available record_sets
    available_record_sets = dataset.record_sets()
    print("Discovered Record Sets:")
    for rs in available_record_sets:
        print(f"Record Set @id: {rs['id']}, name: {rs.get('name', '')}")
else:
    print("Record Sets found in metadata:")
    for rs in record_sets_info:
        print(f"Record Set @id: {rs['id']}, name: {getattr(rs, 'name', '')}")
    available_record_sets = [r['id'] for r in record_sets_info]

# Let's print fields for each record set
fields_by_recordset = {}
for rsid in available_record_sets:
    print(f"--- Record Set: {rsid} ---")
    rs_meta = dataset.record_set_metadata(rsid)
    if rs_meta is not None:
        fields = rs_meta.get('fields', [])
        for f in fields:
            print(f"  Field @id: {f['id']} ({f.get('name', '')}), Type: {f.get('dataType', '')}")
        fields_by_recordset[rsid] = [f['id'] for f in fields]
    else:
        print("  No fields discovered for record set.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all discovered record sets
record_sets_ids = [rs['id'] for rs in dataset.record_sets()]
print("Extracting data from record sets:", record_sets_ids)
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
    else:
        print(f"No records loaded from record set @id: {record_set_id}")

# If at least one dataframe, show its columns and preview
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Sample columns from record set @id: {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded. Check dataset sources or schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Choose a record set with demographic/numeric clinical variables for EDA

# You may need to inspect column names to select appropriate fields
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print("Available Columns in dataframe:")
    print(df.columns.tolist())

    # Try to select likely numeric variables (e.g., age, intervals)
    numeric_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'metastasis' in col.lower() or 'msi' in col.lower()]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field}")
    else:
        numeric_field = df.columns[0]
        print(f"No obvious numeric field found; using: {numeric_field}")

    # Filtering: records with numeric_field > threshold
    try:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    except:
        threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by potential categorical field
    group_candidates = [col for col in df.columns if 'sex' in col.lower() or 'anatomical' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping data by: {group_field}")
        if group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No grouping field available.")
else:
    print("No data loaded; skipping EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Use matplotlib and seaborn to visualize fields from the filtered DataFrame
if dataframes and 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8,6))
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        sns.histplot(filtered_df[numeric_field], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

    # If grouping field exists, show boxplot
    if 'group_field' in locals() and group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        plt.figure(figsize=(10,6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No filtered data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides rich clinicopathological variables for second primary colorectal cancer in cancer survivors.
- Exploration shows information on MSI-H status, anatomical distribution, and demographic features, as well as possible biases and limitations described in the metadata.
- Further clinical analysis can be performed on the loaded record sets. The `mlcroissant` library allows standardized, schema-driven data access for reproducible research.